# Agentic Rag Homework

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# This reads the .env file
load_dotenv() 

# This automatically fetches the GROQ_API_KEY from your environment
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

### Q1. How many lesson pages

In [4]:
### Q1. How many lesson pages
len(documents)

72

### Q2. Indexing and searching
Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

How does the agentic loop keep calling the model until it stops?

What's the filename of the first result?

01-agentic-rag/lessons/03-rag.md

01-agentic-rag/lessons/14-agentic-loop.md

04-evaluation/lessons/13-llm-as-judge.md

06-best-practices/lessons/02-hybrid-search.md

In [5]:
from minsearch import Index
index= Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)
query = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(query,
                            num_results=1)
print(search_results)

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

### the filename is 01-agentic-rag/lessons/14-agentic-loop.md'

### Q3. RAG
Now we will build a RAG assistant on top of this data. Let's use the rag helper script we prepared during the lessons:

wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
RAGBase was written for the FAQ schema (section/question/answer), while our documents have filename and content.

Two solutions are possible:

Implement the RAG flow yourself
Take RAGBase and change the parts related to the FAQ schema - search (to use our index) and build_context
Build a RAG over the index from Q2 and answer the query:

How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

700
7000
70000
700000
We count input tokens instead of price because the cost depends on the model and provider you use, but the size of the prompt we send is the same for everyone.

Most LLM APIs report token usage on the response object (e.g. response.usage.input_tokens / prompt_tokens). We'll read the input tokens from there.

You will need to modify the code for the rag helper to expose the usage.

In the RAG Helper class, llm returns only the text. Modify it to return the whole response, and change rag to return both the answer and usage (as a tuple or create a small dataclass for that).

In [6]:
# Instantiate your helper class
from rag_helper_hw import RAGBase


rag_helper = RAGBase(index=index, model="llama-3.3-70b-versatile",llm_client=client)
query = "How does the agentic loop keep calling the model until it stops?"

# Run a query and unpack the text answer and the usage payload
answer, token_metadata = rag_helper.rag(query)

print("🚀 ANSWER:")
print(answer)
total_tokens=token_metadata.total_tokens
print("\n📊 TOKEN USAGE:")
print(f"Prompt Tokens:     {token_metadata.prompt_tokens}")
print(f"Completion Tokens: {token_metadata.completion_tokens}")
print(f"Total Tokens:      {token_metadata.total_tokens}")

🚀 ANSWER:
I'm ready to help. What's your question about the provided context?

📊 TOKEN USAGE:
Prompt Tokens:     7205
Completion Tokens: 16
Total Tokens:      7221


### Total token is 7221

### Q4. Chunking
The lesson pages are long - some are thousands of characters. Long documents make retrieval less precise: a match deep inside a page still pulls in the whole page. A common fix is chunking: split each page into smaller, overlapping pieces and index those instead.

gitsource has a helper for this: chunk_documents. It uses a sliding window - a window of size characters slides across the text in steps of step characters, and each window position becomes one chunk:

In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [8]:
print(f"Number of chunks created: {len(chunks)}")

Number of chunks created: 295


### Number of created chunks is 295

### Q5. RAG with chunking
Chunking makes each request smaller, because we send a smaller context to the LLM. Let's measure that.

Index the chunks from Q4 (same as before: content as a text field, filename as a keyword field), point your RAG at the chunk index, and answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

about the same
3× fewer
10× fewer
30× fewer

In [9]:
from minsearch import Index
index_chunk= Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index_chunk.fit(chunks)
query = "How does the agentic loop keep calling the model until it stops?"
search_results = index_chunk.search(query,
                            num_results=2)
print(search_results)

[{'start': 4000, 'content': 'while` loop. The loop keeps calling the model until\nit returns a response without any function calls. We also keep an\niteration counter so we can see how many round-trips happened.\n\n```python\nit = 1\n\nwhile True:\n    print(f"iteration #{it}...")\n    has_function_calls = False\n\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=messages,\n        tools=[search_tool],\n    )\n\n    messages.extend(response.output)\n\n    for item in response.output:\n        if item.type == "function_call":\n            print("function_call:", item.name, item.arguments)\n            call_output = make_call(item)\n            messages.append(call_output)\n            has_function_calls = True\n\n        elif item.type == "message":\n            print("ASSISTANT:")\n            print(item.content[0].text)\n\n    it = it + 1\n    if has_function_calls == False:\n        break\n```\n\nThis is the core agent loop. The model reaso

In [10]:
# Instantiate your helper class
from rag_helper_hw import RAGBase


rag_helper = RAGBase(index=index_chunk, model="llama-3.3-70b-versatile",llm_client=client)
query = "How does the agentic loop keep calling the model until it stops?"

# Run a query and unpack the text answer and the usage payload
answer, token_metadata = rag_helper.rag(query)
total_tokens_chunked= token_metadata.total_tokens
print("🚀 ANSWER:")
print(answer)

print("\n📊 TOKEN USAGE:")
print(f"Prompt Tokens chunked:     {token_metadata.prompt_tokens}")
print(f"Completion Tokens chunked: {token_metadata.completion_tokens}")
print(f"Total Tokens chunked:      {token_metadata.total_tokens}")

🚀 ANSWER:
The agentic loop keeps calling the model until it stops by using a `while` loop that continues to execute until a certain condition is met. The condition is that the model returns a response without any function calls. 

In the provided code, this is achieved through the `has_function_calls` flag, which is initially set to `False`. If the model returns a response with a function call, the flag is set to `True`. The loop then checks the value of this flag at the end of each iteration. If the flag is still `False`, it means the model returned a response without any function calls, and the loop breaks, stopping the iteration.

Here's the relevant part of the code:

```python
while True:
    # ...
    has_function_calls = False

    # ...

    for item in response.output:
        if item.type == "function_call":
            # ...
            has_function_calls = True

    # ...

    if has_function_calls == False:
        break
```

This ensures that the loop keeps calling the mo

In [11]:
# Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

ratio_of_token = total_tokens/total_tokens_chunked
print(ratio_of_token)

2.8185011709601873


### So with chunking it is 3 times fewer

### Q6. Turning it into an agent
So far search runs once, with the exact query. Let's make it agentic: give the LLM a search tool and let it decide when (and what) to search. We suggest toyaikit, the small agent library from the module, but you can use anything you like - the OpenAI Agents SDK, PydanticAI, LangChain, or a hand-written loop.

If you go with toyaikit:

uv add toyaikit
Create a search function that uses the chunk index. Give it a type hint and a docstring - most frameworks read them to build the tool schema for you.

Build an agent with your search tool and run it (with toyaikit, the same way as in the ToyAIKit lesson). Use these instructions for the agent (they nudge it to search a few times):

You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.

Ask it:

How does the agentic loop work, and how is it different from plain RAG?

The agent decides on its own when to search and when to answer. Count how many times it called the search tool.

How many times did the agent call search?

Note: the agent decides this itself, so it varies a little between runs - pick the closest option. We measured this with OpenAI gpt-5.4-mini; with a different model or provider the number may differ, so keep that in mind.

0
4
10
20

In [12]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [13]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [15]:
# 2. Set up ToyAIKit Tools and auto-generate the schema via docstrings
agent_tools = Tools()


In [17]:

def search(query: str) -> list:
    """
    Search through the course lessons and indexed text markdown chunks for technical concepts, workflows, or architectural definitions.
    """
    # This print statement acts as your manual counter for the homework
    print(f" 🛠️ [Tool Executed] Framework invoked search with query: '{query}'")
    
    # Execute the lookup against your local text chunk index from Q4/Q5
    results = index_chunk.search(query=query, num_results=5)
    return results


In [18]:
# Register the function
agent_tools.add_tool(search)

In [19]:
# 3. Configure the instructions and runner matching the lesson file
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
""".strip()

homework_question = "How does the agentic loop work, and how is it different from plain RAG?"

In [28]:

# Wrap your client connection and pass the lesson's target model
framework_client = OpenAIClient(model="gpt-5.4-mini")
framework_client.client = openai_client 

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=framework_client
)

In [22]:
# 4. Execute the loop single turn tracking pass
print("Launching the ToyAIKit Framework Loop...\n")
result = runner.loop(prompt=homework_question)

Launching the ToyAIKit Framework Loop...

 🛠️ [Tool Executed] Framework invoked search with query: 'agentic loop RAG lesson agentic loop plain RAG'
 🛠️ [Tool Executed] Framework invoked search with query: 'agentic loop workflow tool use retrieve act observe refine'
 🛠️ [Tool Executed] Framework invoked search with query: 'RAG retrieve generate difference agentic loop'
 🛠️ [Tool Executed] Framework invoked search with query: 'agentic loop definition iterative reasoning with tools'


In [29]:
print("\n FINAL ANSWER:")
# Grab the first text block item from the content list
print(result.all_messages[-1].content[0].text)


 FINAL ANSWER:
The **agentic loop** is an iterative “think → act → observe → repeat” pattern:

1. Send the user request plus the conversation history to the model.
2. The model may return a **tool/function call** instead of a final answer.
3. Your code executes that tool call.
4. You append the tool result back into the message history.
5. You call the model again.
6. Repeat until the model returns a final answer with **no more tool calls**.

In the course, this is described as a `while` loop that keeps calling the LLM, running any returned function calls, and feeding the results back until it stops. The key parts are:

- **Instructions**: what role/behavior the agent should follow
- **Tools**: functions it can call
- **Memory**: the full message history so it remembers what already happened

### How it differs from plain RAG

**Plain RAG** is a fixed pipeline:

```python
search_results = search(question)
user_prompt = build_prompt(question, search_results)
answer = llm(user_prompt)
`

### It called the search tool 4 times